# Predicting next-month arrears

Question: can we predict `arrears_next_month` from what is known on the account at the end of the billing month?

Design decisions (following `checklist.md`):
- **Leakage:** `final_notice_sent` is recorded in the cycle *after* the billing month (see `columns.csv`), so it is not known at prediction time and is dropped. `meter_ref` is an identifier and is dropped (used only for grouping). `billing_month` is used only for splitting.
- **Split:** the model is meant for future rows, and accounts appear in many rows. So the test set is the latest months *and* whole accounts held out of training.
- **CV:** inside train only, with the same structure (earlier months train, later months validate, validation accounts never in the fold's training rows).
- **Candidates** and the selection rule are written down before anything is scored.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
TARGET = 'arrears_next_month'
print(df.shape, 'positive rate', round(df[TARGET].mean(), 4))
print(df.groupby('final_notice_sent')[TARGET].mean().rename('arrears rate by final_notice_sent'))

`final_notice_sent` is almost a proxy for the target (87% vs 1.4% arrears) precisely because it happens *after* the billing month; using it would be leakage.

## 1. Hold out the test set (before any modelling)

In [ ]:
rng = np.random.RandomState(0)
accounts = np.array(sorted(df.meter_ref.unique()))
test_accounts = set(rng.choice(accounts, size=int(0.2 * len(accounts)), replace=False))
months = np.sort(df.billing_month.unique())
cutoff = months[int(len(months) * 0.75)]          # last ~25% of months are the test period
is_test_acc = df.meter_ref.isin(test_accounts)
train = df[~is_test_acc & (df.billing_month < cutoff)].copy()
test = df[is_test_acc & (df.billing_month >= cutoff)].copy()
print('cutoff', pd.Timestamp(cutoff).date(), '| train', train.shape, '| test', test.shape)
print('account overlap:', len(set(train.meter_ref) & set(test.meter_ref)), '| train strictly earlier:', train.billing_month.max() < test.billing_month.min())
print('positive rate train/test:', round(train[TARGET].mean(), 4), round(test[TARGET].mean(), 4))
# rows of train accounts after cutoff, and test accounts before cutoff, are deliberately unused

## 2. Candidates and selection rule (declared before scoring)

Features (known at prediction time): `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

Candidates:
1. **baseline** - always predicts the training prior (trivial).
2. **logreg** - standardised numeric + one-hot tariff, `C` in {0.01, 0.1, 1, 10}.
3. **hgb** - `HistGradientBoostingClassifier`, `max_depth` in {2, 3}, `learning_rate` in {0.03, 0.1}, 150 iterations.

Selection rule: highest mean **PR-AUC** across CV folds (the positive class is ~12%). The decision threshold is then chosen to maximise F1 on the pooled out-of-fold predictions of the chosen model. Nothing is chosen on test.

In [ ]:
FEATS_NUM = ['occupants', 'kwh', 'bill_eur', 'late_payments_12m']; FEATS_CAT = ['tariff']; FEATS = FEATS_NUM + FEATS_CAT
def make(name, **p):
    if name == 'baseline':
        return DummyClassifier(strategy='prior')
    if name == 'logreg':
        pre = ColumnTransformer([('n', StandardScaler(), FEATS_NUM), ('c', OneHotEncoder(handle_unknown='ignore'), FEATS_CAT)])
        return Pipeline([('pre', pre), ('m', LogisticRegression(C=p['C'], max_iter=1000))])
    pre = ColumnTransformer([('n', 'passthrough', FEATS_NUM), ('c', OneHotEncoder(handle_unknown='ignore'), FEATS_CAT)])
    return Pipeline([('pre', pre), ('m', HistGradientBoostingClassifier(max_depth=p['max_depth'], learning_rate=p['learning_rate'], max_iter=150, random_state=0))])
CANDS = [('baseline', {})] + [('logreg', {'C': c}) for c in (0.01, 0.1, 1, 10)] + [('hgb', {'max_depth': d, 'learning_rate': l}) for d in (2, 3) for l in (0.03, 0.1)]
print(len(CANDS), 'candidates')

## 3. Cross-validation inside the training rows
Four expanding-window folds over train months; each fold's validation rows come from a later month block *and* from a held-out quarter of the training accounts, which are removed from that fold's training rows. All fitting (scaling, encoding, model) happens inside the fold via pipelines.

In [ ]:
tr_months = np.sort(train.billing_month.unique())
blocks = np.array_split(tr_months, 5)             # block 0 = initial training window
tr_acc = np.array(sorted(train.meter_ref.unique())); r = np.random.RandomState(1); r.shuffle(tr_acc)
acc_groups = np.array_split(tr_acc, 4)
folds = []
for k in range(4):
    val_mask = train.billing_month.isin(blocks[k + 1]) & train.meter_ref.isin(acc_groups[k])
    tr_mask = train.billing_month.isin(np.concatenate(blocks[:k + 1])) & ~train.meter_ref.isin(acc_groups[k])
    folds.append((np.where(tr_mask)[0], np.where(val_mask)[0]))
    print(k, 'train', tr_mask.sum(), 'val', val_mask.sum(), 'val pos rate', round(train[TARGET][val_mask].mean(), 3))
X, y = train[FEATS], train[TARGET]
rows = []; oof = {}
for i, (n, p) in enumerate(CANDS):
    ap, au, o = [], [], []
    for tri, vai in folds:
        m = make(n, **p).fit(X.iloc[tri], y.iloc[tri]); s = m.predict_proba(X.iloc[vai])[:, 1]
        ap.append(average_precision_score(y.iloc[vai], s)); au.append(roc_auc_score(y.iloc[vai], s)); o.append((vai, s))
    oof[i] = o; rows.append(dict(model=n, params=p, cv_pr_auc=np.mean(ap), cv_roc_auc=np.mean(au)))
cv = pd.DataFrame(rows); cv.sort_values('cv_pr_auc', ascending=False)

## 4. Choose model and threshold on CV only

In [ ]:
best = int(cv.cv_pr_auc.idxmax()); name, params = CANDS[best]
print('chosen:', name, params)
vi = np.concatenate([a for a, _ in oof[best]]); sc = np.concatenate([b for _, b in oof[best]]); yy = y.iloc[vi].values
grid = np.linspace(0.02, 0.9, 89); f1s = [f1_score(yy, sc >= t) for t in grid]
thr = float(grid[int(np.argmax(f1s))])
print('threshold %.2f, OOF F1 %.3f, precision %.3f, recall %.3f' % (thr, max(f1s), precision_score(yy, sc >= thr), recall_score(yy, sc >= thr)))

**Recorded choice (fixed before the test set is touched):** the model, hyperparameters and threshold printed above. Features: the five listed in section 2.

## 5. Refit on all training rows; score the test set once

In [ ]:
final = make(name, **params).fit(X, y)
base = make('baseline').fit(X, y)
Xt, yt = test[FEATS], test[TARGET]
s = final.predict_proba(Xt)[:, 1]; sb = base.predict_proba(Xt)[:, 1]
res = pd.DataFrame({'chosen model': {'ROC AUC': roc_auc_score(yt, s), 'PR AUC': average_precision_score(yt, s),
    'F1 @thr': f1_score(yt, s >= thr), 'precision @thr': precision_score(yt, s >= thr), 'recall @thr': recall_score(yt, s >= thr)},
  'baseline (prior)': {'ROC AUC': roc_auc_score(yt, sb), 'PR AUC': average_precision_score(yt, sb), 'F1 @thr': 0.0, 'precision @thr': 0.0, 'recall @thr': 0.0}})
print('test rows', len(test), 'positive rate', round(yt.mean(), 4)); res.round(3)

## Conclusion
See the test table above (the single headline number). The baseline's PR-AUC equals the test positive rate, so the gap to the chosen model shows the lift attributable to the features. The test set is not reused for any further tuning. Caveat: the test period is only the last quarter of the months and ~20% of accounts, so estimates carry sampling noise; `final_notice_sent` would give a far higher score but is unavailable at prediction time.